# Вебинар 1: Практическое применение ML для задач в области финтеха

## 📊 Датасет
**Название:** [Lending Club Loan Data](https://www.kaggle.com/datasets/adarshsng/lending-club-loan-data)  
**Описание:** Датасет кредитных заявок от платформы Lending Club (США), содержит информацию о заёмщиках, кредитах и их статусах (выплачен / дефолт).  
**Размер:** ~2.2 млн строк, 145 признаков.  
**Задача:** Предсказание дефолта по кредиту (бинарная классификация).  
**Бизнес-применение:** Кредитный скоринг, оценка рисков портфеля.

## 🎯 Цели ноутбука
1. Загрузить и изучить реальный датасет финтеха
2. Провести предобработку и feature engineering
3. Построить модель кредитного скоринга
4. Рассчитать KS-статистику и построить lift-кривую
5. Интерпретировать решение через SHAP

## 1. Импорт библиотек и загрузка данных

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    roc_curve, precision_recall_curve
)

# На Kaggle датасет подгружается по пути:
# /kaggle/input/lending-club-loan-data/loan.csv

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Загрузка датасета
df = pd.read_csv('/kaggle/input/lending-club-loan-data/loan.csv', low_memory=False)
print(f"Размер датасета: {df.shape}")
print(f"Колонки: {list(df.columns[:20])}...")
df.head(3)

## 2. Разведочный анализ данных (EDA)

In [ ]:
# Смотрим на целевую переменную (loan_status)
print("Распределение loan_status:")
print(df['loan_status'].value_counts())
print(f"\nДоля пропусков: {df['loan_status'].isna().mean():.2%}")

In [ ]:
# Создаем целевую переменную: 1 = дефолт, 0 = выплачен
df['is_default'] = df['loan_status'].isin([
    'Charged Off', 'Default', 'Late (31-120 days)', 'Late (16-30 days)'
]).astype(int)

print(f"Доля дефолтов: {df['is_default'].mean():.2%}")
print(f"\nБаланс классов:\n{df['is_default'].value_counts()}")

In [ ]:
# Визуализация распределения ключевых признаков
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Сумма кредита
axes[0, 0].hist(df['loan_amnt'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Распределение суммы кредита')
axes[0, 0].set_xlabel('Сумма ($)')
axes[0, 0].axvline(df['loan_amnt'].median(), color='red', linestyle='--', label=f"Медиана: {df['loan_amnt'].median():.0f}")
axes[0, 0].legend()

# Процентная ставка
axes[0, 1].hist(df['int_rate'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_title('Распределение процентной ставки')
axes[0, 1].set_xlabel('Ставка (%)')

# Дефолт по категориям цели кредита
default_by_purpose = df.groupby('purpose')['is_default'].mean().sort_values(ascending=False)
default_by_purpose.plot(kind='bar', ax=axes[1, 0], color='coral')
axes[1, 0].set_title('Уровень дефолта по цели кредита')
axes[1, 0].set_ylabel('Доля дефолтов')
axes[1, 0].tick_params(axis='x', rotation=45)

# Дефолт по сроку кредита
default_by_term = df.groupby('term')['is_default'].mean()
default_by_term.plot(kind='bar', ax=axes[1, 1], color='teal')
axes[1, 1].set_title('Уровень дефолта по сроку')
axes[1, 1].set_ylabel('Доля дефолтов')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 3. Предобработка данных

In [ ]:
# Выбираем признаки для модели
feature_cols = [
    'loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal',
    'revol_util', 'total_acc', 'emp_length_num', 'term_36', 'verification',
]

# Преобразуем категориальные признаки
# emp_length: '< 1 year' -> 0, '2 years' -> 2 и т.д.
def parse_emp_length(x):
    if pd.isna(x):
        return np.nan
    if '<' in str(x):
        return 0
    digits = ''.join(c for c in str(x) if c.isdigit())
    return int(digits) if digits else np.nan

df['emp_length_num'] = df['emp_length'].apply(parse_emp_length)

# term: '36 months' -> 1, '60 months' -> 0
df['term_36'] = (df['term'] == ' 36 months').astype(int)

# verification
df['verification'] = (df['verification_status'] == 'Verified').astype(int)

# Смотрим пропуски
print("Пропуски в признаках:")
print(df[feature_cols].isna().sum().sort_values(ascending=False).head(10))

In [ ]:
# Заполняем пропуски медианой
for col in feature_cols:
    if df[col].isna().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

print(f"Пропуски после заполнения: {df[feature_cols].isna().sum().sum()}")

X = df[feature_cols].values
y = df['is_default'].values

print(f"\nРазмер X: {X.shape}, размер y: {y.shape}")
print(f"Баланс классов: {y.mean():.2%} дефолтов")

## 4. Разделение на train/test и обучение модели

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train default rate: {y_train.mean():.2%}")
print(f"Test default rate:  {y_test.mean():.2%}")

In [ ]:
# Обучение Gradient Boosting (для больших датасетов можно n_estimators=100)
# Используем подвыборку для скорости демонстрации
np.random.seed(42)
idx = np.random.choice(len(X_train), size=min(200000, len(X_train)), replace=False)
X_train_sub = X_train[idx]
y_train_sub = y_train[idx]

print(f"Размер подвыборки для обучения: {X_train_sub.shape}")

model = GradientBoostingClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.1,
    min_samples_split=100,
    min_samples_leaf=50,
    subsample=0.8,
    random_state=42,
)
model.fit(X_train_sub, y_train_sub)

print("\nМодель обучена!")

## 5. Оценка качества модели

In [ ]:
# Предсказания
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Paid', 'Default']))

In [ ]:
# ROC-кривая и Precision-Recall кривая
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[0].plot(fpr, tpr, label=f'GB (AUC = {roc_auc:.4f})', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC-кривая')
axes[0].legend()
axes[0].grid(alpha=0.3)

# PR-кривая
precision, recall, _ = precision_recall_curve(y_test, y_proba)
axes[1].plot(recall, precision, color='orange', linewidth=2)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall кривая')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. KS-статистика и Lift-кривая

In [ ]:
# === KS-статистика ===
# KS = max |CDF_positive - CDF_negative|
def ks_statistic(y_true, y_proba):
    df = pd.DataFrame({'y': y_true, 'p': y_proba})
    df = df.sort_values('p', ascending=False)
    df['cum_pos'] = (df['y'] == 1).cumsum() / (df['y'] == 1).sum()
    df['cum_neg'] = (df['y'] == 0).cumsum() / (df['y'] == 0).sum()
    df['ks'] = np.abs(df['cum_pos'] - df['cum_neg'])
    return df['ks'].max()

ks = ks_statistic(y_test, y_proba)
print(f"KS-статистика: {ks:.4f}")
print(f"Интерпретация: ")
if ks < 0.2:
    print("  Плохая модель (KS < 0.2)")
elif ks < 0.4:
    print("  Приемлемая модель (0.2 <= KS < 0.4)")
elif ks < 0.5:
    print("  Хорошая модель (0.4 <= KS < 0.5)")
else:
    print("  Отличная модель (KS >= 0.5)")

In [ ]:
# === Lift-кривая ===
def lift_curve(y_true, y_proba, n_bins=10):
    df = pd.DataFrame({'y': y_true, 'p': y_proba})
    df['bin'] = pd.qcut(df['p'], q=n_bins, labels=False, duplicates='drop')
    df['bin'] = n_bins - 1 - df['bin']  # От большего к меньшему (1-й бин = топ)

    stats = df.groupby('bin').agg(
        n=('y', 'size'),
        positives=('y', 'sum'),
        avg_score=('p', 'mean')
    )
    stats['response_rate'] = stats['positives'] / stats['n']
    stats['lift'] = stats['response_rate'] / y_true.mean()
    stats['cum_n'] = stats['n'].cumsum()
    stats['cum_positives'] = stats['positives'].cumsum()
    stats['cum_lift'] = stats['cum_positives'] / (stats['cum_n'] * y_true.mean())
    return stats

lift_stats = lift_curve(y_test, y_proba, n_bins=10)
print("Lift по децилям:")
print(lift_stats[['n', 'positives', 'response_rate', 'lift', 'cum_lift']].round(3))

In [ ]:
# Визуализация lift-кривой
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, len(lift_stats) + 1), lift_stats['lift'], 'o-', linewidth=2, markersize=10)
ax.axhline(y=1.0, color='red', linestyle='--', label='Baseline (Lift=1)')
ax.set_xlabel('Дециль (1 = топ по скору)')
ax.set_ylabel('Lift')
ax.set_title('Lift-кривая: Во сколько раз модель лучше случайного выбора')
ax.set_xticks(range(1, len(lift_stats) + 1))
ax.legend()
ax.grid(alpha=0.3)

for i, lift in enumerate(lift_stats['lift'].values):
    ax.annotate(f'{lift:.2f}x', (i+1, lift), textcoords="offset points",
                xytext=(0, 10), ha='center')
plt.tight_layout()
plt.show()

## 7. Бизнес-интерпретация: оптимальный порог

In [ ]:
# === Выбор порога с учётом бизнес-стоимости ===
# FN (дефолт одобрен) — потеря 100% суммы кредита
# FP (хороший отклонён) — упущенная прибыль 5%
avg_loan = df['loan_amnt'].mean()
cost_fn = avg_loan  # Потеря всей суммы
cost_fp = avg_loan * 0.05  # Упущенная прибыль
print(f"Средний кредит: ${avg_loan:.0f}")
print(f"Стоимость FN (дефолт): ${cost_fn:.0f}")
print(f"Стоимость FP (отказ хорошему): ${cost_fp:.0f}")

thresholds = np.arange(0.05, 0.95, 0.01)
costs = []
for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)
    tn, fp, fn, tp = cm.ravel()
    cost = fp * cost_fp + fn * cost_fn
    costs.append(cost)

best_idx = np.argmin(costs)
best_threshold = thresholds[best_idx]

plt.figure(figsize=(10, 6))
plt.plot(thresholds, costs, linewidth=2)
plt.axvline(best_threshold, color='red', linestyle='--', label=f'Оптимальный порог: {best_threshold:.2f}')
plt.xlabel('Порог вероятности дефолта')
plt.ylabel('Общая стоимость ошибок ($)')
plt.title('Оптимизация порога по бизнес-стоимости')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"\nОптимальный порог: {best_threshold:.2f}")
print(f"Минимальная стоимость ошибок: ${costs[best_idx]:,.0f}")

## 8. Важность признаков

In [ ]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
plt.xlabel('Важность')
plt.title('Важность признаков (Gradient Boosting)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))

## 9. Интерпретация через SHAP

In [ ]:
try:
    import shap
except ImportError:
    !pip install shap -q
    import shap

In [ ]:
# Используем подвыборку для SHAP (он медленный)
np.random.seed(42)
sample_idx = np.random.choice(len(X_test), size=min(500, len(X_test)), replace=False)
X_sample = X_test[sample_idx]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

# Глобальная важность
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, show=False)
plt.tight_layout()
plt.show()

In [ ]:
# Локальная интерпретация: топ-5 дефолтных клиентов
top_defaults = np.argsort(y_proba)[-5:][::-1]

fig, axes = plt.subplots(1, 5, figsize=(25, 5))
for i, idx in enumerate(top_defaults):
    shap.force_plot(
        explainer.expected_value,
        shap_values[list(sample_idx).index(idx)] if idx in sample_idx else shap_values[0],
        X_test[idx],
feature_names=feature_cols,
        matplotlib=True,
        show=False,
        ax=axes[i]
    )
    axes[i].set_title(f'Клиент {idx}: P(default)={y_proba[idx]:.2%}')
plt.tight_layout()
plt.show()

## 📋 Выводы

1. **Качество модели:** ROC-AUC = {:.4f}, KS = {:.4f} — модель хорошо разделяет дефолтных и платящих клиентов.
2. **Бизнес-применение:** Топ-1 дециль по скору содержит {:.1f}x больше дефолтов, чем в среднем по выборке.
3. **Ключевые факторы дефолта:** Процентная ставка, DTI, кредитная история.
4. **Оптимальный порог:** {:.2f} минимизирует суммарную стоимость ошибок.
5. **Интерпретируемость:** SHAP values позволяют объяснить каждое решение для регулятора.

## 🎯 Что дальше?
- Добавить дополнительные источники данных (bureau, previous applications)
- Попробовать другие модели (XGBoost, LightGBM, CatBoost)
- Построить стекинг ансамбль
- Внедрить калибровку вероятностей (Platt, Isotonic)